# Data Scientist Assistant

A personal toolkit for rapid exploratory data analysis, feature engineering, and modeling.  
This notebook demonstrates the full workflow on a sample housing dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ds_utils import (
    profile,
    detect_outliers_iqr,
    plot_distributions,
    plot_correlation_matrix,
    plot_target_vs_features,
    auto_encode_categoricals,
    prepare_data,
    evaluate_regressor,
    quick_cross_val,
    normality_test,
    correlation_test,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
print('Ready!')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/sample_housing.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# One-line data profile: types, nulls, stats, skew, kurtosis
profile(df)

## 2. Missing Values & Outliers

In [ ]:
# Visualize missing data
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) > 0:
    null_counts.plot.bar(color='salmon')
    plt.title('Missing Values per Column')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values!')

# Fill numeric nulls with median
for col in df.select_dtypes(include='number').columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
        print(f'Filled {col} nulls with median')

In [ ]:
# Outlier detection
outlier_flags = detect_outliers_iqr(df)
print('Outlier counts per column:')
print(outlier_flags.sum())

## 3. Exploratory Visualizations

In [ ]:
plot_distributions(df, ['price', 'sqft', 'year_built'])

In [ ]:
plot_correlation_matrix(df)

In [ ]:
plot_target_vs_features(df, target='price', cols=['sqft', 'bedrooms', 'year_built'])

In [ ]:
# Price by neighborhood
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='neighborhood', y='price')
plt.title('Price Distribution by Neighborhood')
plt.tight_layout()
plt.show()

## 4. Statistical Tests

In [ ]:
normality_test(df['price'])

In [ ]:
correlation_test(df['sqft'], df['price'])

## 5. Feature Engineering & Preprocessing

In [ ]:
# Add derived features
df['price_per_sqft'] = df['price'] / df['sqft']
df['house_age'] = 2026 - df['year_built']

# Encode categoricals
df_encoded = auto_encode_categoricals(df.drop(columns=['price_per_sqft']))
print(f'Encoded shape: {df_encoded.shape}')
df_encoded.head()

## 6. Modeling — Predict House Price

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X_train, X_test, y_train, y_test, scaler = prepare_data(
    df_encoded, target='price', test_size=0.2, scale=True
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Linear Regression
lr = LinearRegression().fit(X_train, y_train)
print('=== Linear Regression ===')
evaluate_regressor(lr, X_test, y_test)

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
print('=== Random Forest ===')
evaluate_regressor(rf, X_test, y_test)

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42).fit(X_train, y_train)
print('=== Gradient Boosting ===')
evaluate_regressor(gb, X_test, y_test)

In [ ]:
# Cross-validation comparison
from sklearn.model_selection import cross_val_score

X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

print('=== 5-Fold CV (neg_mean_absolute_error) ===')
for name, model in [('LinearReg', lr), ('RandomForest', rf), ('GradientBoosting', gb)]:
    scores = cross_val_score(model, X_all, y_all, cv=5, scoring='neg_mean_absolute_error')
    print(f'{name:20s} MAE = {-scores.mean():,.0f} (+/- {scores.std():,.0f})')

## 7. Feature Importance

In [ ]:
importances = pd.Series(gb.feature_importances_, index=X_train.columns).sort_values(ascending=True)
importances.plot.barh(figsize=(8, 6), color='steelblue')
plt.title('Feature Importance (Gradient Boosting)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

---

## Quick-Start Guide

To use this toolkit on **your own data**:

```python
# 1. Load your data
df = pd.read_csv('your_data.csv')

# 2. Profile it
profile(df)

# 3. Visualize
plot_distributions(df)
plot_correlation_matrix(df)

# 4. Preprocess
df_encoded = auto_encode_categoricals(df)
X_train, X_test, y_train, y_test, scaler = prepare_data(df_encoded, target='your_target')

# 5. Model & evaluate
model = GradientBoostingRegressor().fit(X_train, y_train)
evaluate_regressor(model, X_test, y_test)
```

All utility functions live in `src/ds_utils.py` — import what you need!